<h1 style="color: coral;">Procesamiento de Datos de Coches de Segunda Mano</h1>
  
Este notebook procesa datos de diferentes fuentes para combinarlos y generar un resumen.
Incluye pasos para normalizar datos, buscar coincidencias de modelos similares, y crear un archivo final.

Integrantes:

- `Pablo Gandia`
- `Nouh Khouyi Etbar`
- `Pablo Alpuente Tabasco`

### Funcion del programa:

Este programa procesa y combina datos de coches de segunda mano provenientes de tres páginas web:

- MotorES
- OcasionPlus
- AutoScout24

Principales características:
- Identifica y combina únicamente los modelos que aparecen en todas las páginas web scrapeadas.
- Genera un resumen con los valores promedio de Precio, Kilómetros y Año para cada modelo, segmentados por página web.
- Permite comparar modelos entre las diferentes páginas web, facilitando el análisis de precios, kilometraje y años de matriculación.
Objetivo del programa:
El objetivo es ayudar a identificar tendencias y discrepancias en los precios y características de los coches entre las tres plataformas, proporcionando una herramienta útil para realizar comparativas y tomar decisiones informadas.



#### Importación de librerías 
Se utilizan `pandas` para manipulación de datos y `difflib` para buscar coincidencias aproximadas.

In [1]:
import pandas as pd
import difflib

#### Definición de la función `encontrar_coincidencia`
Esta función busca la coincidencia más cercana para un modelo de coche en una lista existente.
Se utiliza para asegurar que nombres similares (pero con diferencias menores) sean agrupados.

Es útil porque los nombres de los modelos pueden tener ligeras variaciones en las distintas bases de datos (por ejemplo, "Ford Fiesta" vs "Fiesta Ford").

In [2]:
def encontrar_coincidencia(modelo, modelos_existentes):
    modelo = modelo.lower().strip() # Normalización del modelo:
    match = difflib.get_close_matches(modelo, modelos_existentes, n=3, cutoff=0.6) # Búsqueda de coincidencias aproximadas (hasta 3)
    return match[0] if match else modelo # deevuelve la más similar (match[0]) o si no se encuentra ninguna coincidencia, devuelve el modelo original
    

#### Función `cargar_y_normalizar_datos`
Procesa un archivo CSV para normalizar y estructurar los datos. Sus principales tareas son:

- Limpieza de datos: Elimina espacios innecesarios y asegura consistencia en los nombres de columnas.  
- Renombrar columnas: Adapta las columnas a nombres estándar para facilitar el análisis.  
- Combinación de columnas: Crea un campo Modelo unificado al combinar columnas relevantes (si es necesario).  
- Normalización de campos: Ajusta formatos en Año (extrae años válidos) y Precio (limpia valores no numéricos).  
- Unificación de modelos: Busca coincidencias aproximadas para estandarizar nombres de modelos.  
- Etiqueta de origen: Agrega una columna con el nombre de la página de origen de los datos.  

In [3]:
def cargar_y_normalizar_datos(ruta, columnas_relevantes, renombrar_columnas, columnas_combinadas, pagina, modelos_existentes, index_col=None):
    df = pd.read_csv(ruta, index_col=index_col)  # Carga archivo
    df.columns = df.columns.str.strip()  # Limpia los nombres de las columnas

    if columnas_combinadas:  # Asegura que todos los datos tengan un campo Modelo consistente
        df['Modelo'] = df[columnas_combinadas[0]].str.strip() + ' ' + df[columnas_combinadas[1]].str.strip()
    else:
        df['Modelo'] = df['Modelo'].str.strip()

    df = df[columnas_relevantes]
    df.rename(columns=renombrar_columnas, inplace=True)  # Estándariza nombres

    if 'Año' in df.columns:
        df['Año'] = df['Año'].astype(str).str.extract(r'(\d{4})')[0]
        df['Año'] = pd.to_numeric(df['Año'], errors='coerce')

    df['Modelo'] = df['Modelo'].apply(lambda x: encontrar_coincidencia(x, modelos_existentes))

    if 'Precio' in df.columns:
        df['Precio'] = df['Precio'].replace(r'[^\d.]', '', regex=True).astype(float)
    
    if 'Kilómetros' in df.columns:
        if pagina == 'OcasionPlus':
            df['Kilómetros'] = df['Kilómetros'].astype(str).str.replace('.', '', regex=False).replace('nan', '0').astype(float)
        elif pagina == 'MotorES':
            df['Kilómetros'] = df['Kilómetros'].astype(float)
        else:
            df['Kilómetros'] = df['Kilómetros'].astype(str).str.replace('.', '', regex=False).astype(float)

        # Convertir a enteros (asegurarse de manejar NaN)
        df['Kilómetros'] = df['Kilómetros'].fillna(0).astype(int)

    df['Página'] = pagina
    return df

#### Función principal `main`
La función main coordina todo el flujo, desde la carga de datos hasta la generación y exportación del resumen final, asegurando que los datos sean consistentes y fáciles de analizar.


1. **Carga y normalización de datos**:
   - Procesa datos de tres fuentes (**MotorES**, **OcasionPlus**, y **AutoScout24**) usando la función `cargar_y_normalizar_datos`.
   - Combina estos datos en un único DataFrame, asegurando consistencia en los campos.
2. **Filtrado de modelos**:
   - Identifica modelos presentes en las tres páginas de origen.
   - Filtra el DataFrame para incluir solo estos modelos.
3. **Creación de un resumen**:
   - Genera una tabla pivotada que calcula promedios de `Precio`, `Kilómetros`, y `Año` por modelo y página.
4. **Limpieza y ajustes finales**:
   - Redondea precios y kilómetros para mayor legibilidad.
   - Convierte años en enteros, eliminando valores nulos.
5. **Exportación de resultados**:
   - Guarda la tabla resumen en un archivo CSV llamado `resumen.csv` y informa al usuario cuando la fusión y el guardado se completan.


In [4]:

def main():
    modelos_existentes = []

    # Cargar y normalizar datos de diferentes fuentes
    motor_es = cargar_y_normalizar_datos(
        'MotorES.csv',
        ['Marca', 'Modelo', 'Precio', 'Kilómetros', 'Año Matriculación'],
        {'Año Matriculación': 'Año'},
        ['Marca', 'Modelo'],
        'MotorES',
        modelos_existentes,
        index_col=0
    )
    modelos_existentes.extend(motor_es['Modelo'].unique())

    ocasion_plus = cargar_y_normalizar_datos(
        'OcasionPlusData.csv',
        ['Brand', 'Name', 'Price', 'KM', 'Year'],
        {'Brand': 'Marca', 'Name': 'Modelo', 'Price': 'Precio', 'KM': 'Kilómetros', 'Year': 'Año'},
        ['Brand', 'Name'],
        'OcasionPlus',
        modelos_existentes
    )
    modelos_existentes.extend(ocasion_plus['Modelo'].unique())

    autoscout24 = cargar_y_normalizar_datos(
        'ScrapAutoScout24.csv',
        ['Coche', 'Precio', 'Kilometraje (Km)', 'Año'],
        {'Coche': 'Modelo', 'Kilometraje (Km)': 'Kilómetros'},
        None,
        'AutoScout24',
        modelos_existentes
    )
    modelos_existentes.extend(autoscout24['Modelo'].unique())

    # Combinar todos los datos
    combined_df = pd.concat([motor_es, ocasion_plus, autoscout24], ignore_index=True)
    combined_df = combined_df.dropna(subset=['Precio', 'Kilómetros', 'Año'])

    # Filtrar modelos que estén presentes en todas las fuentes
    model_page_counts = combined_df.groupby('Modelo')['Página'].nunique()
    valid_models = model_page_counts[model_page_counts == 3].index
    filtered_df = combined_df[combined_df['Modelo'].isin(valid_models)]

    # Crear un resumen
    summary = filtered_df.pivot_table(
        index='Modelo',
        columns='Página',
        values=['Precio', 'Kilómetros', 'Año'],
        aggfunc='mean'
    )

    summary.columns = [f"{col}_{page}" for col, page in summary.columns]

    # Redondear y convertir a los tipos adecuados
    for col in summary.columns:
        if 'Precio' in col:
            summary[col] = summary[col].round(1)
        elif 'Año' in col:
            summary[col] = summary[col].dropna().astype(int)
        else: 
            summary[col] = summary[col].astype(int)

    # Guardar el resultado
    summary.reset_index().to_csv('resumen.csv', index=False)
    print("Fusión completada. Archivo guardado como 'resumen.csv'.")

if __name__ == "__main__":
    main()


Fusión completada. Archivo guardado como 'resumen.csv'.
